<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 24 · Building a Market and Broker for Trading

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the Chapter 24 examples in an interactive format. It
loads the `engine` package, compares historical and simulated tick feeds,
places paper orders, inspects broker receipts, and runs a small session loop.


### How to Use This Notebook
- Run the cells from top to bottom the first time.
- The setup cell switches into the project root so relative imports still
  work.
- The examples stay close to the chapter and expose the main `engine` API.


### Notebook Setup
Move to the project root first so that the notebook can reuse the same
relative paths and local packages as the chapter scripts.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

## Importing the Engine Package
Bring the main market, broker, and session objects into the notebook so the
remaining examples can stay compact.


In [ ]:
import json

import pandas as pd

from engine import (
    OrderRequest,
    PaperBroker,
    Tick,
    TradingSession,
    build_feed,
    estimate_gbm_parameters,
)


## Historical and Simulated Tick Feeds
Create one historical replay feed and one GBM-based simulated feed for the
same symbol, then inspect the first emitted ticks from each feed.


In [ ]:
historical_feed = build_feed(symbol="EURUSD", mode="historical")
simulated_feed = build_feed(
    symbol="EURUSD", mode="simulated", periods=8, seed=42
)

historical_ticks = pd.DataFrame(
    [next(iter(build_feed(symbol="EURUSD", mode="historical"))).to_dict()]
)
simulated_ticks = pd.DataFrame(
    [
        tick.to_dict()
        for tick in build_feed(
            symbol="EURUSD", mode="simulated", periods=3, seed=42
        )
    ]
)

historical_ticks


In [ ]:
simulated_ticks


## Estimating GBM Parameters
The simulated feed can estimate drift and volatility from a historical price
series before generating synthetic ticks.


In [ ]:
prices = historical_feed.prices.iloc[-60:]
mu, sigma = estimate_gbm_parameters(prices)
pd.Series({"mu": mu, "sigma": sigma}).round(6)


## Market Orders, Receipts, and Account State
Route one market order through the paper broker, then inspect the returned
receipt and the updated account snapshot.


In [ ]:
first_tick = next(iter(build_feed(symbol="EURUSD", mode="historical")))

broker = PaperBroker(initial_cash=50_000.0, account_id="NOTEBOOK-24")
broker.on_tick(first_tick)
receipt = broker.place_order(
    OrderRequest(
        symbol="EURUSD",
        side="buy",
        quantity=10.0,
        client_order_id="ENTRY-001",
        meta={"source": "notebook"},
    )
)
receipt


In [ ]:
snapshot = broker.get_account_snapshot()
pd.DataFrame(snapshot["positions"])


In [ ]:
pd.Series({
    "cash": snapshot["cash"],
    "equity": snapshot["equity"],
    "realized_pnl": snapshot["realized_pnl"],
    "unrealized_pnl": snapshot["unrealized_pnl"],
}).round(6)


## Stop-Loss Orders
Register a stop-loss order, force a lower synthetic tick, and verify that the
broker triggers the protective exit automatically.


In [ ]:
stop_receipt = broker.place_stop_loss(
    symbol="EURUSD",
    stop_price=round(first_tick.bid * 0.98, 6),
)
stop_receipt


In [ ]:
trigger_tick = Tick(
    timestamp=first_tick.timestamp + pd.offsets.BusinessDay(1),
    symbol="EURUSD",
    bid=round(first_tick.bid * 0.97, 6),
    ask=round(first_tick.ask * 0.97, 6),
    source="synthetic_trigger",
)
triggered = broker.on_tick(trigger_tick)
triggered


In [ ]:
broker.get_account_snapshot()


## Running a Small Session Loop
Tie a feed and broker together in a `TradingSession`, then let a minimal
strategy open one position and attach one stop-loss order.


In [ ]:
def demo_strategy(session: TradingSession, tick: Tick) -> None:
    if not session.receipts:
        session.place_market_order(
            symbol=tick.symbol,
            side="buy",
            quantity=10.0,
            client_order_id="ENTRY-001",
            meta={"strategy": "chapter24_notebook"},
        )
        session.place_stop_loss(
            symbol=tick.symbol,
            stop_price=round(tick.bid * 0.98, 6),
        )

session = TradingSession(
    feed=build_feed(symbol="EURUSD", mode="historical"),
    broker=PaperBroker(initial_cash=50_000.0, account_id="SESSION-24"),
)
history = session.run(demo_strategy)
history[["bid", "ask", "mid", "cash", "equity", "position_quantity"]].head()


## Inspecting a JSON Receipt
The returned receipt objects are plain dictionaries and can therefore be turned
into formatted JSON directly in the notebook.


In [ ]:
print(json.dumps(receipt, indent=2))
